<a href="https://colab.research.google.com/github/rperezen/intro-to-ML/blob/main/hw_5_pt_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

# ==========================================
# 3.a Preprocessing & Setup (11 Features)
# ==========================================
df = pd.read_csv('Housing.csv')

# Map 'yes'/'no' string columns to binary 1 and 0
binary_vars = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for var in binary_vars:
    if var in df.columns:
        df[var] = df[var].apply(lambda x: 1 if str(x).lower() == 'yes' else (0 if str(x).lower() == 'no' else x))

# Use all 11 input features
features = ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom',
            'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea']

X = df[features].values
y = df['price'].values.reshape(-1, 1)

# Standardize features to prevent gradient explosion
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_scaled = (X - X_mean) / X_std

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# 80/20 Train-Validation Split (Sequential)
split_idx = int(0.8 * len(X))
X_train, y_train = X_tensor[:split_idx], y_tensor[:split_idx]
X_val, y_val = X_tensor[split_idx:], y_tensor[split_idx:]

def model(X, w, b):
    return torch.matmul(X, w) + b

def loss_fn(preds, targets):
    return torch.mean((preds - targets) ** 2)

# ==========================================
# 3.b Training Loop & Hyperparameter Tuning
# ==========================================
epochs = 5000
learning_rates = [0.1, 0.01, 0.001, 0.0001]

best_val_loss = float('inf')
best_params = {}

for lr in learning_rates:
    print(f"\n--- Training with Learning Rate: {lr} ---")

    # Initialize 12 parameters: 11 weights and 1 bias
    w = torch.zeros((11, 1), dtype=torch.float32, requires_grad=True)
    b = torch.zeros(1, dtype=torch.float32, requires_grad=True)

    optimizer = optim.SGD([w, b], lr=lr)

    for epoch in range(1, epochs + 1):
        preds = model(X_train, w, b)
        train_loss = loss_fn(preds, y_train)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            with torch.no_grad():
                val_preds = model(X_val, w, b)
                val_loss = loss_fn(val_preds, y_val)

            print(f"Epoch {epoch} | Train Loss: {train_loss.item():.2e} | Val Loss: {val_loss.item():.2e}")

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_params = {'lr': lr, 'w': w.detach().clone(), 'b': b.detach().clone()}

print("\n==========================================")
print(" BEST MODEL PARAMETERS (11 FEATURES)")
print("==========================================")
print(f"Optimal Learning Rate: {best_params['lr']}")
print(f"Lowest Final Validation Loss: {best_val_loss:.2e}")
print("\nThe Twelve Identified Parameters:")
for i, feature in enumerate(features):
    print(f" - Weight ({feature}): {best_params['w'][i].item():.2f}")
print(f" - Bias (Intercept): {best_params['b'].item():.2f}")


--- Training with Learning Rate: 0.1 ---
Epoch 500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 1000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 1500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 2000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 2500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 3000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 3500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 4000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 4500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 5000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12

--- Training with Learning Rate: 0.01 ---
Epoch 500 | Train Loss: 1.19e+12 | Val Loss: 1.15e+12
Epoch 1000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 1500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 2000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 2500 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 3000 | Train Loss: 1.19e+12 | Val Loss: 1.16e+12
Epoch 3500 | Train Loss: 1.19e+12 | V